In [162]:
# Imports

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.tuner import Tuner
from ray import tune
from ray.tune.integration.pytorch_lightning import TuneReportCallback
import os
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

## Getting Data

In [163]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("muhammadmusharraf444/used-car-price-prediction-dataset")
data_path = os.path.join(path, "quikr_car.csv")

print("Path to dataset files:", data_path)

Path to dataset files: /home/franio/.cache/kagglehub/datasets/muhammadmusharraf444/used-car-price-prediction-dataset/versions/1/quikr_car.csv


In [164]:
data_df = pd.read_csv(data_path)

data_df.dropna(inplace=True)
data_df = data_df[data_df.Price != 'Ask For Price' ]
data_df.Price = data_df.Price.str.replace(',', '').astype(float)

data_df.kms_driven = data_df.kms_driven.str.strip('kms').str.replace(',', '', regex=False).str.strip().astype(float)
data_df.year = data_df.year.astype(int)
data_df = pd.get_dummies(data_df, columns=['company', 'fuel_type'], dtype=int)

data_df.drop(['name'], axis=1, inplace=True)

data_df

,year,Price,kms_driven,company_Audi,company_BMW,company_Chevrolet,company_Datsun,company_Fiat,company_Force,company_Ford,...,company_Nissan,company_Renault,company_Skoda,company_Tata,company_Toyota,company_Volkswagen,company_Volvo,fuel_type_Diesel,fuel_type_LPG,fuel_type_Petrol
0,2007,80000.0,45000.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,2006,425000.0,40.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,2014,325000.0,28000.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,2014,575000.0,36000.0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
6,2012,175000.0,41000.0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,2011,270000.0,50000.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
885,2009,110000.0,30000.0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
886,2009,300000.0,132000.0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
888,2018,260000.0,27000.0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0


# Classes

In [165]:
class MyDataset(Dataset):
 def __init__(self, data, outputs):
  """Inicjalizacja - przygotowanie struktury danych"""
  self.data = data
  self.outputs = outputs

 def __len__(self):
  """Zwraca liczbę próbek w datasecie"""
  return len(self.data)

 def __getitem__(self, idx):
  """Zwraca próbkę o indeksie idx"""
  x = self.data[idx]
  y = self.outputs[idx]
  return x, y

class MyDataModule(pl.LightningDataModule):
    def __init__(self, X: np.ndarray, y: np.ndarray, batch_size: int = 32):
        super().__init__()
        self.X = X
        self.y = y
        self.batch_size = batch_size

        self.X_scaler = StandardScaler()
        self.y_scaler = StandardScaler()

        # Placeholders to track state across stages
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None

    def setup(self, stage: str = None):
        # Guard clause: Prevents re-splitting and re-scaling data on subsequent stage calls
        if self.train_dataset is not None:
            return


        X_train_val, X_test, y_train_val, y_test = train_test_split(
            self.X, self.y, test_size=0.1, random_state=42
        )


        X_train, X_val, y_train, y_val = train_test_split(
            X_train_val, y_train_val, test_size=0.2, random_state=42
        )


        X_train = self.X_scaler.fit_transform(X_train)
        y_train = self.y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()

        X_val = self.X_scaler.transform(X_val)
        y_val = self.y_scaler.transform(y_val.reshape(-1, 1)).ravel()

        X_test = self.X_scaler.transform(X_test)
        y_test = self.y_scaler.transform(y_test.reshape(-1, 1)).ravel()


        self.train_dataset = MyDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).float())
        self.val_dataset = MyDataset(torch.from_numpy(X_val).float(), torch.from_numpy(y_val).float())
        self.test_dataset = MyDataset(torch.from_numpy(X_test).float(), torch.from_numpy(y_test).float())

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size)

In [166]:
import torch
import torch.nn as nn
import pytorch_lightning as pl
from torchmetrics.regression import R2Score

class LitModel(pl.LightningModule):
    def __init__(self, input_size, n_hidden, hidden_size, output_dim, lr=0.001):
        super().__init__()
        self.save_hyperparameters() # Accessible via self.hparams.lr, etc.

        # Network Layers
        self.input_layer = nn.Linear(input_size, hidden_size)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_size, hidden_size) for _ in range(n_hidden)])
        self.output_layer = nn.Linear(hidden_size, output_dim)

        # Loss and Metrics (Kept on device automatically)
        self.criterion = nn.MSELoss()
        self.r2_metric = R2Score()

    def forward(self, x):
        x = torch.relu(self.input_layer(x))
        for layer in self.hidden_layers:
            x = torch.relu(layer(x))
        x = self.output_layer(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)

        # Ensure shapes match without breaking on batch_size=1
        loss = self.criterion(y_hat.view_as(y), y)

        self.log('train_loss', loss, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x).view_as(y)

        loss = self.criterion(y_hat, y)
        r2 = self.r2_metric(y_hat, y)

        # prog_bar=True displays these beautifully in the terminal terminal
        self.log('val_loss', loss, prog_bar=True, on_epoch=True)
        self.log('val_r2', r2, prog_bar=True, on_epoch=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x).view_as(y)

        loss = self.criterion(y_hat, y)
        r2 = self.r2_metric(y_hat, y)

        self.log('test_loss', loss, on_epoch=True)
        self.log('test_r2', r2, on_epoch=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

# Optimization

In [167]:
def train_model(config, X, y):
    model = LitModel(input_size=X.shape[1],
        n_hidden=config['n_hidden'],
        hidden_size=config['hidden_size'],
        output_dim=1,
        lr=config['lr']
    )

    datamodule = MyDataModule(X=X, y=y, batch_size=8)

    trainer = Trainer(
        max_epochs=20,
        log_every_n_steps=5,
        enable_checkpointing=False,
        callbacks=[TuneReportCallback({'loss': 'val_loss'}, on='validation_end')],
    )

    trainer.fit(model, datamodule)

# Ray Tune search
trainable_with_data = tune.with_parameters(train_model, X=X, y=y)

analysis = tune.run(
    trainable_with_data,
    config={
        'n_hidden': tune.choice([3,6,8,10]),
        'hidden_size': tune.choice([32, 64, 128, 256, 512]),
        'lr': tune.loguniform(1e-4, 5e-1),
    },
    num_samples=20
)

2026-06-15 14:31:20,926	INFO tune.py:615 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


(train_model pid=44533) /home/franio/Desktop/Deep_Learning_Labs/venv/lib/python3.11/site-packages/ray/tune/integration/pytorch_lightning.py:198: `ray.tune.integration.pytorch_lightning.TuneReportCallback` is deprecated. Use `ray.tune.integration.pytorch_lightning.TuneReportCheckpointCallback` instead.
(train_model pid=44533) GPU available: False, used: False
(train_model pid=44533) TPU available: False, using: 0 TPU cores
(train_model pid=44533) 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
(train_model pid=44533) 
(train_model pid=44533)   | Name          | Type       | Params | Mode  | FLOPs
(train_model pid=44533) -------------------------------------------------------------
(train_model pid=44533) 0 | input_layer   | Linear     | 992    | train | 0    
(train_model pid=44533) 1 | hidden_layers | Modul

Epoch 0:  22%|██▏       | 16/74 [00:00<00:01, 57.64it/s, v_num=0]
                                                                           
Epoch 0:   0%|          | 0/74 [00:00<?, ?it/s]
Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Epoch 0:  38%|███▊      | 28/74 [00:00<00:00, 63.89it/s, v_num=0]


(train_model pid=44523) 
(train_model pid=44523) 0 | input_layer   | Linear     | 7.9 K  | train | 0    
(train_model pid=44528) 
(train_model pid=44532) 


Epoch 0:  16%|█▌        | 12/74 [00:00<00:01, 43.21it/s, v_num=0]
                                                                           


...

(train_model pid=45250) `Trainer.fit` stopped: `max_epochs=20` reached.


(train_model pid=44529) 
(train_model pid=44529) 
(train_model pid=44529) 
(train_model pid=44529) 
Epoch 12:  93%|█████████▎| 69/74 [00:04<00:00, 16.64it/s, v_num=0, val_loss=0.967, val_r2=-0.694] [repeated 8x across cluster]
Validation: |          | 0/? [00:00<?, ?it/s] [repeated 8x across cluster]
Validation DataLoader 0: 100%|██████████| 19/19 [00:00<00:00, 138.60it/s] [repeated 6x across cluster]
(train_model pid=45344) 
(train_model pid=45344) 
(train_model pid=45344) 
Epoch 12: 100%|██████████| 74/74 [00:04<00:00, 16.91it/s, v_num=0, val_loss=0.967, val_r2=-0.694]
Validation: |          | 0/? [00:00<?, ?it/s] [repeated 8x across cluster]
Epoch 14: 100%|██████████| 74/74 [00:02<00:00, 25.72it/s, v_num=0, val_loss=0.967, val_r2=-0.689]
(train_model pid=45344) 
Epoch 15: 100%|██████████| 74/74 [00:03<00:00, 21.42it/s, v_num=0, val_loss=0.966, val_r2=-0.686]
(train_model pid=45344) 
(train_model pid=45344) 
(train_model pid=45344) 
(train_model pid=45239) 
(train_model pid=45344) 
(

(train_model pid=45344) `Trainer.fit` stopped: `max_epochs=20` reached. [repeated 2x across cluster]


Validation: |          | 0/? [00:00<?, ?it/s] [repeated 6x across cluster]
Epoch 14:  96%|█████████▌| 71/74 [00:03<00:00, 20.62it/s, v_num=0, val_loss=2.48e+26, val_r2=-8.7e+26]
Validation: |          | 0/? [00:00<?, ?it/s] [repeated 2x across cluster]
Validation DataLoader 0:   0%|          | 0/19 [00:00<?, ?it/s]
(train_model pid=45239) 
(train_model pid=45239) 
Validation DataLoader 0:   0%|          | 0/19 [00:00<?, ?it/s]
(train_model pid=45239) 
(train_model pid=45239) 
Epoch 16:  82%|████████▏ | 61/74 [00:02<00:00, 22.10it/s, v_num=0, val_loss=2.48e+26, val_r2=-8.7e+26] [repeated 6x across cluster]
Validation: |          | 0/? [00:00<?, ?it/s] [repeated 4x across cluster]
Validation DataLoader 0:   0%|          | 0/19 [00:00<?, ?it/s]
(train_model pid=45239) 
(train_model pid=45239) 
Epoch 17:  80%|███████▉  | 59/74 [00:02<00:00, 19.78it/s, v_num=0, val_loss=2.48e+26, val_r2=-8.69e+26] [repeated 2x across cluster]
Validation: |          | 0/? [00:00<?, ?it/s] [repeated 2x across

2026-06-15 14:34:04,898	INFO tune.py:1001 -- Wrote the latest version of all result files and experiment state to '/home/franio/ray_results/train_model_2026-06-15_14-31-20' in 0.0088s.
2026-06-15 14:34:04,905	INFO tune.py:1033 -- Total run time: 163.98 seconds (163.95 seconds for the tuning loop).


In [168]:
print(analysis.get_best_config(metric='loss', mode='min'))

{'n_hidden': 3, 'hidden_size': 256, 'lr': 0.0009268147769941901}


# Model Training

In [169]:
config = analysis.get_best_config(metric='loss', mode='min')
model = LitModel(input_size=X.shape[1], output_dim=1, **config)

# Stop jeśli val_loss nie poprawia się przez 5 epochs
early_stop_callback = EarlyStopping(
  monitor='val_loss',
  patience=5,
  mode='min',
  verbose=True
)

model_path = os.path.join('Models', 'Task2')
os.makedirs(model_path, exist_ok=True)


# Zapisz best model (według val_loss)
checkpoint_callback = ModelCheckpoint(
  dirpath=model_path,
  filename='model-{epoch:02d}-{val_loss:.2f}',
  monitor='val_loss',
  mode='min',
  save_top_k=3, # 3 najlepsze
  save_last=True # ostatni checkpoint
)

trainer = Trainer(
  callbacks=[early_stop_callback,checkpoint_callback],
  max_epochs=100,
  accelerator='gpu',
  devices=1
 )

trainer.fit(model, datamodule=datamodule)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type       | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | input_layer   | Linear     | 7.9 K  | train | 0    
1 | hidden_layers | ModuleList | 197 K  | train | 0    
2 | output_layer  | Linear     | 257    | train | 0    
3 | criterion     | MSELoss    | 0      | train | 0    
4 | r2_metric     | R2Score    | 0      | train | 0    
-------------------------------------------------------------
205 K     Trainable params
0         Non-trainable params
205 K     Total params
0.822     Total estimated model params size (MB)
8         Modules in train mode
0         Modules in eval

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved. New best score: 654770438144.000


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_loss did not improve in the last 5 records. Best score: 563665436672.000. Signaling Trainer to stop.


In [170]:
trainer.test(model, datamodule=datamodule)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss             138060595200.0
         test_r2           -0.10326997190713882
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 138060595200.0, 'test_r2': -0.10326997190713882}]

## Test Prediction for one datapoint

In [171]:
scaler = module.X_scaler
sample = scaler.transform(X[42].reshape(1, -1))
sample_t = torch.from_numpy(sample).float()

model.eval()
out = model(sample_t)
pred = module.y_scaler.inverse_transform(out.detach().numpy()).ravel()[0]

print(y[42])
print(pred)

284999.0
250525420.0


Task 3 is part of second project